**SparkSession**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, round as spark_round

spark = SparkSession.builder \
    .appName("Tugas6-ETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/24 20:19:00 WARN Utils: Your hostname, xcel resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/24 20:19:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 20:19:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 20:19:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


**A. EXTRACT**

Ketiga sumber dibaca menjadi tiga DataFrame terpisah. CSV memakai header=True dan inferSchema=True. JSON Lines dibaca dengan spark.read.json() sehingga skemanya terdeteksi otomatis.

In [2]:
# EXTRACT — membaca tiga sumber data
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

for nama, df in [("Transaksi", df_trx), ("Produk", df_produk), ("Ulasan", df_ulasan)]:
    print(f"===== {nama}: {df.count()} baris =====")
    df.printSchema()

===== Transaksi: 5000 baris =====
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

===== Produk: 30 baris =====
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

===== Ulasan: 3500 baris =====
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



**B. TRANSFORM — Penggabungan**

Pilihan join dan alasannya:
- Transaksi > Produk (product_id),	menggunakan **inner** karena, setiap transaksi pasti punya produk yang valid, jadi tidak ada baris yang hilang. Inner join menegaskan bahwa hanya transaksi dengan produk valid yang lolos.
- Transaksi > Ulasan (order_id) menggunakan **left** kaarena, tidak semua transaksi memiliki ulasan. Left join menjaga seluruh transaksi. Inner join akan membuang transaksi tanpa ulasan (1.500 baris) dan membuat data bias.

Kolom total_pendapatan dihitung dari unit_terjual x harga.

In [3]:
# TRANSFORM (B) — join tiga sumber + kolom total_pendapatan
df_gabung = (
    df_trx
    .join(df_produk, on="product_id", how="inner")   # transaksi -> produk
    .join(df_ulasan, on="order_id", how="left")      # transaksi -> ulasan
    .withColumn("total_pendapatan", col("unit_terjual") * col("harga"))
)

print("Jumlah baris setelah join :", df_gabung.count())
print("Jumlah baris transaksi awal:", df_trx.count())

df_gabung.select(
    "order_id", "product_id", "nama_produk", "kategori",
    "unit_terjual", "harga", "total_pendapatan", "rating"
).show(5)

Jumlah baris setelah join : 5000
Jumlah baris transaksi awal: 5000
+--------+----------+-----------+------------+------------+------+----------------+------+
|order_id|product_id|nama_produk|    kategori|unit_terjual| harga|total_pendapatan|rating|
+--------+----------+-----------+------------+------------+------+----------------+------+
|     TX0|        21|  Produk-21|  Elektronik|           2| 25000|           50000|     1|
|     TX1|         8|   Produk-8|     Fashion|           6|250000|         1500000|  NULL|
|     TX2|        25|  Produk-25|  Elektronik|           4| 25000|          100000|     2|
|     TX3|         3|   Produk-3|     Fashion|           4| 75000|          300000|     5|
|     TX4|        19|  Produk-19|Rumah Tangga|           6| 75000|          450000|  NULL|
+--------+----------+-----------+------------+------------+------+----------------+------+
only showing top 5 rows



**C. TRANSFORM — Penanganan Data Kosong dan Pengayaan**

Urutan langkah sangat penting. Kolom ada_ulasan dibuat lebih dulu dari status null pada rating. Baru setelah itu null diisi 0. Jika urutannya dibalik, semua rating sudah 0 dan informasi "ada ulasan atau tidak" tidak bisa lagi dibedakan dengan benar.

In [4]:
# TRANSFORM (C) — langkah 1: buat ada_ulasan SEBELUM na.fill()
# Nilainya diturunkan dari data (rating tidak null), bukan diisi manual.
df_gabung = df_gabung.withColumn("ada_ulasan", col("rating").isNotNull())

# Cek jumlah null sebelum diisi
print("Rating null sebelum na.fill():", df_gabung.filter(col("rating").isNull()).count())

# langkah 2: isi rating kosong dengan 0 menggunakan na.fill()
df_etl = df_gabung.na.fill(0, subset=["rating"])

print("Rating null setelah na.fill():", df_etl.filter(col("rating").isNull()).count())

# Validasi ada_ulasan
df_etl.groupBy("ada_ulasan").count().show()

df_etl.select(
    "order_id", "nama_produk", "kategori", "unit_terjual",
    "harga", "total_pendapatan", "rating", "ada_ulasan"
).show(5)

Rating null sebelum na.fill(): 1500
Rating null setelah na.fill(): 0
+----------+-----+
|ada_ulasan|count|
+----------+-----+
|      true| 3500|
|     false| 1500|
+----------+-----+

+--------+-----------+------------+------------+------+----------------+------+----------+
|order_id|nama_produk|    kategori|unit_terjual| harga|total_pendapatan|rating|ada_ulasan|
+--------+-----------+------------+------------+------+----------------+------+----------+
|     TX0|  Produk-21|  Elektronik|           2| 25000|           50000|     1|      true|
|     TX1|   Produk-8|     Fashion|           6|250000|         1500000|     0|     false|
|     TX2|  Produk-25|  Elektronik|           4| 25000|          100000|     2|      true|
|     TX3|   Produk-3|     Fashion|           4| 75000|          300000|     5|      true|
|     TX4|  Produk-19|Rumah Tangga|           6| 75000|          450000|     0|     false|
+--------+-----------+------------+------------+------+----------------+------+---------

**Mengapa nilai 0 masuk akal untuk "belum ada ulasan"?**
Nilai 0 dipilih karena skala rating di data ini hanya 1 sampai 5, jadi angka 0 tidak mungkin tertukar dengan penilaian asli dari pelanggan. Angka 0 juga secara alami terbaca sebagai "belum ada penilaian". Kalau diisi nilai lain, misalnya 3 atau rata-rata rating, kita malah seolah-olah mengarang pendapat pelanggan yang sebenarnya tidak pernah diberikan. Perlu diingat juga, nilai 0 akan menurunkan rata-rata kalau ikut dihitung, jadi untuk analisis kepuasan pelanggan data difilter dulu dengan ada_ulasan = True.

**D. LOAD**

Hasil akhir disimpan ke HDFS dalam format Parquet, dipartisi berdasarkan kategori.

In [5]:
# LOAD — siapkan direktori di HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas6

In [7]:
# LOAD — tulis ke HDFS: Parquet, dipartisi berdasarkan kategori
path_hasil = "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"

df_etl.write.mode("overwrite").partitionBy("kategori").parquet(path_hasil)
print("Pipeline ETL selesai — hasil tersimpan di HDFS.")

[Stage 45:>                                                         (0 + 1) / 1]

Pipeline ETL selesai — hasil tersimpan di HDFS.


Verifikasi struktur folder di HDFS:

In [8]:
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

-rw-r--r--   3 xcell supergroup          0 2026-09-24 20:28 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - xcell supergroup          0 2026-09-24 20:28 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 xcell supergroup      16364 2026-09-24 20:28 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-136bc091-70c0-4d1f-9eb9-8f52f65c4af9.c000.snappy.parquet
drwxr-xr-x   - xcell supergroup          0 2026-09-24 20:28 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 xcell supergroup      11134 2026-09-24 20:28 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-136bc091-70c0-4d1f-9eb9-8f52f65c4af9.c000.snappy.parquet
drwxr-xr-x   - xcell supergroup          0 2026-09-24 20:28 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 xcell supergroup      10393 2026-09-24 20:28 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-136bc091-70c0-4d1f-9eb9-8f52f65c4af9.c000.snappy.parquet
drwxr-xr-x   - xcell sup

Baca kembali dari HDFS sebagai bukti data tersimpan utuh:

In [9]:
df_final = spark.read.parquet(path_hasil)

print("Jumlah baris hasil akhir :", df_final.count())
print("Jumlah baris transaksi   :", df_trx.count())
df_final.printSchema()

Jumlah baris hasil akhir : 5000
Jumlah baris transaksi   : 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- harga: long (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- total_pendapatan: long (nullable = true)
 |-- ada_ulasan: boolean (nullable = true)
 |-- kategori: string (nullable = true)



**E. INSIGHT AKHIR**

Persentase transaksi berulasan per kategori dihitung dari data yang dibaca kembali dari HDFS (df_final) dengan Spark SQL.

In [11]:
df_final.createOrReplaceTempView("hasil_etl")

spark.sql('''
    SELECT
        kategori,
        COUNT(*) AS total_transaksi,
        SUM(CASE WHEN ada_ulasan THEN 1 ELSE 0 END) AS transaksi_berulasan,
        ROUND(100.0 * SUM(CASE WHEN ada_ulasan THEN 1 ELSE 0 END) / COUNT(*), 2) AS persen_ulasan
    FROM hasil_etl
    GROUP BY kategori
    ORDER BY persen_ulasan ASC
''').show()

[Stage 56:======================================>                   (2 + 1) / 3]

+------------+---------------+-------------------+-------------+
|    kategori|total_transaksi|transaksi_berulasan|persen_ulasan|
+------------+---------------+-------------------+-------------+
|     Makanan|            536|                369|        68.84|
|   Kesehatan|            961|                662|        68.89|
|     Fashion|           1035|                726|        70.14|
|Rumah Tangga|            815|                575|        70.55|
|  Elektronik|           1653|               1168|        70.66|
+------------+---------------+-------------------+-------------+



Cara setara dengan DataFrame API:

In [12]:
from pyspark.sql.functions import avg

df_final.groupBy("kategori").agg(
    count("*").alias("total_transaksi"),
    spark_round(avg(col("ada_ulasan").cast("int")) * 100, 2).alias("persen_ulasan")
).orderBy("persen_ulasan").show()

[Stage 59:===================>                                      (1 + 2) / 3]

+------------+---------------+-------------+
|    kategori|total_transaksi|persen_ulasan|
+------------+---------------+-------------+
|     Makanan|            536|        68.84|
|   Kesehatan|            961|        68.89|
|     Fashion|           1035|        70.14|
|Rumah Tangga|            815|        70.55|
|  Elektronik|           1653|        70.66|
+------------+---------------+-------------+



Jawaban: Kategori dengan persentase transaksi berulasan paling rendah adalah Makanan (68,84%, yaitu 369 dari 536 transaksi).

**Interpretasi Bisnis**

Kategori Makanan punya persentase transaksi berulasan paling rendah, yaitu 68,84%, meskipun selisihnya sangat tipis dengan kategori lain yang semuanya berada di kisaran 69% sampai 71%. Artinya, tidak ada kategori yang benar-benar bermasalah, tetapi sekitar 3 dari 10 pembeli belum meninggalkan ulasan, dan Makanan adalah yang paling perlu didorong. Bagi tim marketing, ulasan penting karena bisa menjadi bukti sosial yang memengaruhi keputusan calon pembeli baru. Karena itu, pengingat ulasan setelah barang diterima, misalnya dengan voucher atau poin loyalitas, sebaiknya diprioritaskan untuk kategori Makanan.

In [13]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
